# Task 101: pyilc_cmb — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: CMB component separation using internal linear combination (ILC)

**Paper**: pyilc: Internal Linear Combination for CMB foreground removal (no formal paper found)
**Repository**: https://github.com/jcolinhill/pyilc

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 29.51 dB
- **SSIM**: 0.7575
- **Evaluation**: CMB map recovery via ILC (PSNR + SSIM + CC)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
pyilc_cmb - CMB Component Separation via Internal Linear Combination
=====================================================================
From multi-frequency sky maps, extract the CMB signal using ILC.

Physics:
  - Simulate CMB (Gaussian random field) + synchrotron + dust at 6 frequencies
  - Forward: d_ν = a_ν * CMB + foregrounds(ν) + noise
  - Inverse: ILC weights  w = (a^T C^{-1} a)^{-1} C^{-1} a
  - Recovered CMB = w^T d
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`gaussian_random_field`, `synchrotron_template`, `dust_template`, `ilc_weights`, `main`

In [ ]:
# ====================================================================
# 1. Simulate sky components
# ====================================================================
def gaussian_random_field(N, power_law_index=-2.0, rms=1.0):
    """Generate a 2D Gaussian random field with power-law power spectrum."""
    kx = np.fft.fftfreq(N, d=1.0 / N)
    ky = np.fft.fftfreq(N, d=1.0 / N)
    KX, KY = np.meshgrid(kx, ky)
    K = np.sqrt(KX**2 + KY**2)
    K[0, 0] = 1.0  # avoid division by zero
    power = K ** power_law_index
    power[0, 0] = 0.0  # zero mean
    phases = np.random.uniform(0, 2 * np.pi, (N, N))
    amplitudes = np.sqrt(power) * np.exp(1j * phases)
    field = np.fft.ifft2(amplitudes).real
    field = field / field.std() * rms
    return field

def synchrotron_template(N, freq_ghz, ref_freq=0.408):
    """Synchrotron emission: power law ∝ ν^{-3.0} from ref_freq GHz."""
    template = gaussian_random_field(N, power_law_index=-2.5, rms=200.0)
    return template * (freq_ghz / ref_freq) ** (-3.0)

def dust_template(N, freq_ghz, ref_freq=545.0):
    """Thermal dust emission: modified blackbody ∝ ν^{1.5} × B_ν."""
    template = gaussian_random_field(N, power_law_index=-2.5, rms=150.0)
    return template * (freq_ghz / ref_freq) ** 1.5

# ====================================================================
# 2. ILC Inverse
# ====================================================================
def ilc_weights(data):
    """
    Compute ILC weights: w = (a^T C^{-1} a)^{-1} C^{-1} a
    where a = [1, 1, ..., 1] (CMB has unit response at all frequencies
    in thermodynamic temperature units).
    
    Parameters
    ----------
    data : (N_freq, N_pix_total) — flattened frequency maps
    
    Returns
    -------
    w : (N_freq,) ILC weights
    """
    n_freq = data.shape[0]
    a = np.ones(n_freq)
    
    # Covariance matrix
    C = np.cov(data)
    
    # Regularise slightly for numerical stability
    C += 1e-10 * np.eye(n_freq)
    
    C_inv = np.linalg.inv(C)
    
    # ILC weights
    w = C_inv @ a / (a @ C_inv @ a)
    return w

# ====================================================================
# 5. Main
# ====================================================================
def main():
    print("=" * 60)
    print("Task 101: CMB Component Separation (pyilc_cmb)")
    print("=" * 60)

    t0 = time.time()

    # Simulate
    print("\n[1] Simulating multi-frequency sky maps ...")
    cmb_gt, data = simulate_observations(N_PIX, FREQS_GHZ)

    # ILC recovery
    print("[2] Applying ILC ...")
    cmb_rec, weights = recover_cmb(data)

    elapsed = time.time() - t0
    print(f"    Elapsed: {elapsed:.1f} s")
    print(f"    ILC weights: {weights}")
    print(f"    Sum of weights: {weights.sum():.6f}")

    # Metrics
    print("[3] Computing metrics ...")
    psnr, ssim_val, cc, rmse = compute_metrics(cmb_gt, cmb_rec)

    print(f"    PSNR = {psnr:.2f} dB")
    print(f"    SSIM = {ssim_val:.4f}")
    print(f"    CC   = {cc:.4f}")
    print(f"    RMSE = {rmse:.2f} μK")

    metrics = {
        "PSNR": float(psnr),
        "SSIM": float(ssim_val),
        "CC": float(cc),
        "RMSE": float(rmse),
    }

    # Save
    print("[4] Saving outputs ...")
    for d in [RESULTS_DIR, ASSETS_DIR]:
        np.save(os.path.join(d, "gt_output.npy"), cmb_gt)
        np.save(os.path.join(d, "recon_output.npy"), cmb_rec)
        with open(os.path.join(d, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)

    # Plot
    print("[5] Plotting ...")
    plot_results(FREQS_GHZ, data, cmb_gt, cmb_rec, weights, metrics)

    print(f"\n{'='*60}")
    print("Task 101 COMPLETE")
    print(f"{'='*60}")
    return metrics

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `create_cmb`, `simulate_observations`

In [ ]:
def create_cmb(N, rms=CMB_RMS):
    """Simulate CMB as Gaussian random field ~ l^(-2) power spectrum."""
    return gaussian_random_field(N, power_law_index=-2.0, rms=rms)

def simulate_observations(N, freqs):
    """
    Generate multi-frequency sky maps.
    
    Returns
    -------
    cmb_gt : (N, N) ground truth CMB map
    data   : (N_freq, N, N) observed maps
    """
    cmb_gt = create_cmb(N)
    data = np.zeros((len(freqs), N, N))
    
    for i, nu in enumerate(freqs):
        fg_sync = synchrotron_template(N, nu)
        fg_dust = dust_template(N, nu)
        noise = np.random.normal(0, NOISE_RMS[i], (N, N))
        # CMB has flat SED in thermodynamic temperature units: a_ν = 1
        data[i] = cmb_gt + fg_sync + fg_dust + noise
    
    return cmb_gt, data

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `recover_cmb`

In [ ]:
def recover_cmb(data):
    """
    Apply ILC to recover CMB map.
    
    Parameters
    ----------
    data : (N_freq, N, N) multi-frequency maps
    
    Returns
    -------
    cmb_rec : (N, N) recovered CMB
    w : (N_freq,) ILC weights
    """
    n_freq, ny, nx = data.shape
    data_flat = data.reshape(n_freq, -1)
    
    w = ilc_weights(data_flat)
    
    cmb_flat = w @ data_flat
    cmb_rec = cmb_flat.reshape(ny, nx)
    
    return cmb_rec, w

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `plot_results`

In [ ]:
# ====================================================================
# 3. Metrics
# ====================================================================
def compute_metrics(cmb_gt, cmb_rec):
    """PSNR, SSIM, CC of recovered CMB."""
    mse = np.mean((cmb_gt - cmb_rec)**2)
    data_range = cmb_gt.max() - cmb_gt.min()
    psnr = 10.0 * np.log10(data_range**2 / mse) if mse > 0 else 100.0
    
    ssim_val = ssim(cmb_gt, cmb_rec, data_range=data_range)
    
    cc = float(np.corrcoef(cmb_gt.ravel(), cmb_rec.ravel())[0, 1])
    
    rmse = float(np.sqrt(mse))
    
    return psnr, ssim_val, cc, rmse

# ====================================================================
# 4. Visualization
# ====================================================================
def plot_results(freqs, data, cmb_gt, cmb_rec, weights, metrics_dict):
    """Multi-panel figure."""
    fig = plt.figure(figsize=(18, 12))
    
    # Row 1: selected frequency maps (3 of 6)
    sel = [0, 2, 5]  # 30, 70, 217 GHz
    for idx, si in enumerate(sel):
        ax = fig.add_subplot(3, 3, idx + 1)
        vmax = np.percentile(np.abs(data[si]), 99)
        ax.imshow(data[si], cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        ax.set_title(f"{freqs[si]:.0f} GHz", fontsize=10)
        ax.axis('off')
    
    # Row 2: GT CMB, recovered CMB, residual
    vmax_cmb = np.percentile(np.abs(cmb_gt), 99)
    
    ax = fig.add_subplot(3, 3, 4)
    ax.imshow(cmb_gt, cmap='RdBu_r', vmin=-vmax_cmb, vmax=vmax_cmb)
    ax.set_title("GT CMB", fontsize=10)
    ax.axis('off')
    
    ax = fig.add_subplot(3, 3, 5)
    ax.imshow(cmb_rec, cmap='RdBu_r', vmin=-vmax_cmb, vmax=vmax_cmb)
    ax.set_title("Recovered CMB (ILC)", fontsize=10)
    ax.axis('off')
    
    ax = fig.add_subplot(3, 3, 6)
    residual = cmb_gt - cmb_rec
    vmax_res = np.percentile(np.abs(residual), 99)
    ax.imshow(residual, cmap='RdBu_r', vmin=-vmax_res, vmax=vmax_res)
    ax.set_title(f"Residual (RMS={np.std(residual):.1f} μK)", fontsize=10)
    ax.axis('off')
    
    # Row 3 left: ILC weights
    ax = fig.add_subplot(3, 3, 7)
    ax.bar(range(len(freqs)), weights, color='steelblue')
    ax.set_xticks(range(len(freqs)))
    ax.set_xticklabels([f"{f:.0f}" for f in freqs], fontsize=8)
    ax.set_xlabel("Frequency (GHz)")
    ax.set_ylabel("Weight")
    ax.set_title("ILC Weights")
    ax.axhline(0, color='k', lw=0.5)
    
    # Row 3 middle: power spectra comparison
    ax = fig.add_subplot(3, 3, 8)
    ps_gt = np.abs(np.fft.fft2(cmb_gt))**2
    ps_rec = np.abs(np.fft.fft2(cmb_rec))**2
    k = np.arange(1, N_PIX // 2)
    kx = np.fft.fftfreq(N_PIX, d=1.0) * N_PIX
    ky = np.fft.fftfreq(N_PIX, d=1.0) * N_PIX
    KX, KY = np.meshgrid(kx, ky)
    K = np.sqrt(KX**2 + KY**2)
    
    cl_gt = np.zeros(len(k))
    cl_rec = np.zeros(len(k))
    for i, ki in enumerate(k):
        mask = (K >= ki - 0.5) & (K < ki + 0.5)
        if mask.sum() > 0:
            cl_gt[i] = ps_gt[mask].mean()
            cl_rec[i] = ps_rec[mask].mean()
    
    ax.loglog(k, cl_gt, 'b-', label='GT', lw=1.5)
    ax.loglog(k, cl_rec, 'r--', label='ILC', lw=1.5)
    ax.set_xlabel("Multipole ℓ")
    ax.set_ylabel("C_ℓ")
    ax.set_title("Angular Power Spectrum")
    ax.legend(fontsize=8)
    
    # Row 3 right: metrics text
    ax = fig.add_subplot(3, 3, 9)
    ax.axis('off')
    txt = (f"PSNR = {metrics_dict['PSNR']:.2f} dB\n"
           f"SSIM = {metrics_dict['SSIM']:.4f}\n"
           f"CC   = {metrics_dict['CC']:.4f}\n"
           f"RMSE = {metrics_dict['RMSE']:.2f} μK\n"
           f"\nΣ weights = {weights.sum():.6f}")
    ax.text(0.1, 0.5, txt, fontsize=14, family='monospace',
            transform=ax.transAxes, verticalalignment='center')
    ax.set_title("Metrics Summary")
    
    plt.tight_layout()
    for path in [os.path.join(RESULTS_DIR, "vis_result.png"),
                 os.path.join(ASSETS_DIR, "vis_result.png")]:
        fig.savefig(path, dpi=150)
    plt.close(fig)

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
metrics = main()

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **pyilc_cmb**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=29.51 dB, SSIM=0.7575)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: pyilc: Internal Linear Combination for CMB foreground removal (no formal paper found)
- Repository: https://github.com/jcolinhill/pyilc
